### بسم الله الرحمن الرحیم

### در ابتدا باید فایل زیر را به اسم زیر ذخیره کنیم
###   MRH_make_probe_filelists.py

In [ ]:
### MRH_make_probe_filelists.py
import argparse
import csv
import random
import re
from pathlib import Path
from collections import Counter


VCTK_PATTERN = re.compile(
    r"^(?P<speaker>p\d+)_(?P<utterance>\d+)_mic(?P<mic>\d+)\.flac$",
    flags=re.IGNORECASE,
)


def parse_vctk_file(wav_path):
    """
    Parse a VCTK filename such as:
        p260_006_mic1.flac
    """
    match = VCTK_PATTERN.match(wav_path.name)

    if match is None:
        return None

    return {
        "path": str(wav_path.resolve()),
        "speaker": match.group("speaker").lower(),
        "utterance": match.group("utterance"),
        "microphone": int(match.group("mic")),
    }


def write_path_list(output_path, samples):
    """
    Write a FreeVC-style file list containing one absolute path per line.
    No DUMMY prefix is added.
    """
    with output_path.open("w", encoding="utf-8", newline="\n") as file:
        for sample in samples:
            file.write(sample["path"] + "\n")


def write_manifest(output_path, samples):
    """
    Write metadata required by speaker/content probing.
    """
    fieldnames = [
        "path",
        "speaker",
        "utterance",
        "microphone",
        "split",
    ]

    with output_path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(
            file,
            fieldnames=fieldnames,
            delimiter="\t",
        )
        writer.writeheader()
        writer.writerows(samples)


def main():
    parser = argparse.ArgumentParser(
        description="Create VCTK file lists for layer-wise probing."
    )

    parser.add_argument(
        "--source_dir",
        type=str,
        required=True,
        help="VCTK wav48_silence_trimmed directory",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="./filelists/probing",
        help="Directory used to save output file lists",
    )
    parser.add_argument(
        "--prefix",
        type=str,
        default="VCTK_probe",
        help="Output filename prefix",
    )
    parser.add_argument(
        "--microphone",
        type=int,
        default=1,
        help="VCTK microphone number to include; mic1 is recommended",
    )
    parser.add_argument(
        "--val_per_speaker",
        type=int,
        default=5,
        help="Number of validation utterances per speaker",
    )
    parser.add_argument(
        "--test_per_speaker",
        type=int,
        default=10,
        help="Number of test utterances per speaker",
    )
    parser.add_argument(
        "--train_ratio",
        type=float,
        default=1.0,
        help="Fraction of remaining utterances used for training",
    )
    parser.add_argument(
        "--max_speakers",
        type=int,
        default=None,
        help="Optional maximum number of speakers",
    )
    parser.add_argument(
        "--max_files_per_speaker",
        type=int,
        default=None,
        help="Optional maximum number of files selected per speaker",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed",
    )

    args = parser.parse_args()

    if not 0.0 < args.train_ratio <= 1.0:
        raise ValueError("--train_ratio must be in the interval (0, 1].")

    source_dir = Path(args.source_dir).expanduser()

    if not source_dir.exists():
        raise FileNotFoundError(
            f"Source directory does not exist:\n{source_dir}"
        )

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Scanning VCTK directory:\n{source_dir}")

    speaker_files = {}

    # VCTK silence-trimmed files normally use FLAC.
    for wav_path in sorted(source_dir.rglob("*.flac")):
        
        print ("MRH wav_path= ",wav_path)
        metadata = parse_vctk_file(wav_path)

        if metadata is None:
            continue

        if metadata["microphone"] != args.microphone:
            continue

        speaker = metadata["speaker"]
        speaker_files.setdefault(speaker, []).append(metadata)

    speakers = sorted(speaker_files)

    if args.max_speakers is not None:
        speakers = speakers[:args.max_speakers]

    if not speakers:
        raise RuntimeError(
            "No matching VCTK FLAC files were found. "
            "Check source_dir and microphone selection."
        )

    train_samples = []
    val_samples = []
    test_samples = []
    skipped_speakers = []

    minimum_required = (
        args.val_per_speaker
        + args.test_per_speaker
        + 1
    )

    for speaker_index, speaker in enumerate(speakers):
        print("MRH speaker = ", speaker)
        samples = list(speaker_files[speaker])

        # A separate deterministic RNG is used for every speaker.
        speaker_rng = random.Random(args.seed + speaker_index)
        speaker_rng.shuffle(samples)

        if args.max_files_per_speaker is not None:
            samples = samples[:args.max_files_per_speaker]

        if len(samples) < minimum_required:
            skipped_speakers.append((speaker, len(samples)))
            continue

        val_end = args.val_per_speaker
        test_end = val_end + args.test_per_speaker

        val_part = samples[:val_end]
        test_part = samples[val_end:test_end]
        remaining_train = samples[test_end:]

        number_of_train_samples = max(
            1,
            int(len(remaining_train) * args.train_ratio),
        )
        train_part = remaining_train[:number_of_train_samples]

        for sample in train_part:
            sample["split"] = "train"

        for sample in val_part:
            sample["split"] = "val"

        for sample in test_part:
            sample["split"] = "test"

        train_samples.extend(train_part)
        val_samples.extend(val_part)
        test_samples.extend(test_part)

    # Shuffle global list order without changing split membership.
    global_rng = random.Random(args.seed)
    global_rng.shuffle(train_samples)
    global_rng.shuffle(val_samples)
    global_rng.shuffle(test_samples)

    all_samples = train_samples + val_samples + test_samples

    train_path = output_dir / f"{args.prefix}_train.txt"
    val_path = output_dir / f"{args.prefix}_val.txt"
    test_path = output_dir / f"{args.prefix}_test.txt"
    manifest_path = output_dir / f"{args.prefix}_manifest.tsv"

    write_path_list(train_path, train_samples)
    write_path_list(val_path, val_samples)
    write_path_list(test_path, test_samples)
    write_manifest(manifest_path, all_samples)

    split_counts = Counter(sample["split"] for sample in all_samples)
    included_speakers = sorted(
        {sample["speaker"] for sample in all_samples}
    )

    print("\nDataset statistics")
    print("-" * 55)
    print(f"Included speakers : {len(included_speakers)}")
    print(f"Skipped speakers  : {len(skipped_speakers)}")
    print(f"Train samples     : {split_counts['train']}")
    print(f"Validation samples: {split_counts['val']}")
    print(f"Test samples      : {split_counts['test']}")
    print(f"Total samples     : {len(all_samples)}")
    print(f"Microphone        : mic{args.microphone}")
    print("-" * 55)

    if skipped_speakers:
        print("\nSkipped speakers:")
        for speaker, count in skipped_speakers:
            print(f"  {speaker}: only {count} usable files")

    print("\nGenerated files:")
    print(f"  {train_path.resolve()}")
    print(f"  {val_path.resolve()}")
    print(f"  {test_path.resolve()}")
    print(f"  {manifest_path.resolve()}")

    print("\nFile lists created successfully.")


if __name__ == "__main__":
    main()


### ۳. اقدام توصیه‌شده: بازسازی split سخت‌تر

In [ ]:
!python MRH_make_probe_filelists.py --source_dir "/mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed" --output_dir "./filelists/probing_hard" --prefix "VCTK_100spk_20utt" --microphone 1 --max_speakers 100 --max_files_per_speaker 20 --val_per_speaker 3 --test_per_speaker 5 --train_ratio 1.0 --seed 42

Scanning VCTK directory:
/mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_001_mic1.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_001_mic2.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_002_mic1.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_002_mic2.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_003_mic1.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_003_mic2.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_004_mic1.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_004_mic2.flac
MRH wav_path=  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_silence_trimmed/p225/p225_005_mic1.flac
MRH wav_path= 

In [9]:
import pandas as pd
from pathlib import Path

MANIFEST_PATH = Path("./filelists/probing_hard/VCTK_100spk_20utt_manifest.tsv")

manifest = pd.read_csv(MANIFEST_PATH, sep="\t")

required_columns = {"path", "speaker", "utterance", "microphone", "split"}
missing_columns = required_columns - set(manifest.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# تصحیح تبدیل نوع داده‌ها
manifest["speaker"] = manifest["speaker"].astype(str)
manifest["utterance"] = manifest["utterance"].astype(str)
manifest["split"] = manifest["split"].astype(str)

print("--- Data Summary ---")
print(manifest.head(5))
print("\nSplit counts:\n", manifest["split"].value_counts())
print("\nUnique speakers:", manifest["speaker"].nunique())

# بررسی معتبر بودن مسیر فایل‌ها
missing_files = sum(not Path(p).exists() for p in manifest["path"])
print(f"Missing audio files count: {missing_files}")


--- Data Summary ---
                                                path speaker utterance  \
0  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_s...    p263        59   
1  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_s...    p301       155   
2  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_s...    p267        57   
3  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_s...    p262       330   
4  /mnt/d/PHD/MyDatasets/VCTK-Corpus-0.92/wav48_s...    p295       295   

   microphone  split  
0           1  train  
1           1  train  
2           1  train  
3           1  train  
4           1  train  

Split counts:
 split
train    1200
test      500
val       300
Name: count, dtype: int64

Unique speakers: 100
Missing audio files count: 0


### گام ۱: بررسی و تأیید مانیفست جدید (100 گوینده)

In [12]:
import pandas as pd
from pathlib import Path

# مسیر مانیفست 100 گوینده‌ای جدید
HARD_MANIFEST_PATH = Path("./filelists/probing_hard/VCTK_100spk_20utt_manifest.tsv")

if not HARD_MANIFEST_PATH.exists():
    raise FileNotFoundError(f"فایل مانیفست در مسیر {HARD_MANIFEST_PATH} یافت نشد!")

manifest_hard = pd.read_csv(HARD_MANIFEST_PATH, sep="\t")

manifest_hard["speaker"] = manifest_hard["speaker"].astype(str)
manifest_hard["utterance"] = manifest_hard["utterance"].astype(str)
manifest_hard["split"] = manifest_hard["split"].astype(str)

print("--- Hard Dataset Summary ---")
print(f"Total Audio Samples: {len(manifest_hard)}")
print("\nSplit Distribution:")
print(manifest_hard["split"].value_counts())
print("\nUnique Speakers:", manifest_hard["speaker"].nunique())

# چک کردن موجود بودن تمام فایل‌های صوتی
missing_files = sum(not Path(p).exists() for p in manifest_hard["path"])
print(f"\nMissing Audio Files Count: {missing_files}")


--- Hard Dataset Summary ---
Total Audio Samples: 2000

Split Distribution:
split
train    1200
test      500
val       300
Name: count, dtype: int64

Unique Speakers: 100

Missing Audio Files Count: 0


### گام ۲: استخراج Embeddingها روی مانیفست جدید

In [16]:
from pathlib import Path
import torch
import torchaudio
from tqdm.auto import tqdm

EMBEDDING_DIR_HARD = Path("./probing_outputs/hard_speaker_embeddings")
EMBEDDING_DIR_HARD.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16000

def load_audio_for_probe(audio_path, target_sr=TARGET_SR):
    waveform, sample_rate = torchaudio.load(audio_path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if sample_rate != target_sr:
        waveform = torchaudio.functional.resample(waveform, sample_rate, target_sr)
    return waveform

def pool_hidden_states(hidden_state):
    hidden_state = hidden_state.float()
    mean_vec = hidden_state.mean(dim=0)
    std_vec = hidden_state.std(dim=0, unbiased=False)
    return {
        "mean": mean_vec,
        "mean_std": torch.cat([mean_vec, std_vec], dim=0)
    }

def extract_layerwise_utterance_embeddings(audio_path):
    audio = load_audio_for_probe(audio_path)
    layerwise_embeddings = {}

    for model_name in loaded_models.keys():
        hidden_states = extract_hidden_states(model_name, audio)
        
        pooled_mean = []
        pooled_mean_std = []
        
        for hs in hidden_states:
            pools = pool_hidden_states(hs)
            pooled_mean.append(pools["mean"].cpu())
            pooled_mean_std.append(pools["mean_std"].cpu())
            
        layerwise_embeddings[model_name] = {
            "mean": pooled_mean,
            "mean_std": pooled_mean_std
        }

    return layerwise_embeddings

# استخراج داده‌های مانیفست 100 گوینده‌ای
metadata_records_hard = []

for row_index, row in tqdm(
    manifest_hard.iterrows(),
    total=len(manifest_hard),
    desc="Extracting 100-Speaker Embeddings",
):
    audio_path = row["path"]
    try:
        embeddings = extract_layerwise_utterance_embeddings(audio_path)
        output_file = EMBEDDING_DIR_HARD / f"{row_index:06d}.pt"

        torch.save(
            {
                "path": audio_path,
                "speaker": row["speaker"],
                "utterance": row["utterance"],
                "microphone": int(row["microphone"]),
                "split": row["split"],
                "embeddings": embeddings,
            },
            output_file,
        )

        metadata_records_hard.append(
            {
                "row_index": row_index,
                "embedding_path": str(output_file),
                "path": audio_path,
                "speaker": row["speaker"],
                "utterance": row["utterance"],
                "split": row["split"],
            }
        )
    except Exception as error:
        print(f"\nFailed processing {audio_path}: {error}")

embedding_index_hard = pd.DataFrame(metadata_records_hard)
embedding_index_hard_path = EMBEDDING_DIR_HARD / "embedding_index_hard.tsv"
embedding_index_hard.to_csv(embedding_index_hard_path, sep="\t", index=False)

print(f"\nSuccessfully extracted and saved {len(embedding_index_hard)} embeddings.")


Extracting 100-Speaker Embeddings: 100%|███████████████████████████████████████████| 2000/2000 [05:59<00:00,  5.57it/s]



Successfully extracted and saved 2000 embeddings.
